In [38]:
import pandas as pd
import plotly.graph_objects as go

In [39]:
def load_impacto_mensal(proposta:str)->pd.DataFrame:
    fname = f'impacto_mensal_{proposta}.csv'
    return pd.read_csv(fname, index_col=0, sep=';')

In [40]:
impactos_mensais ={
    'Equiparação com a tabela dos AMCI' : load_impacto_mensal('amci'),
    'Reajuste pelo IPC-Fipe - jun. 2016' : load_impacto_mensal('ipc'),
    'Reajuste pelo IPC-Fipe - jun. 2021' : load_impacto_mensal('ipc_nunes'),
    "Situação atual" : load_impacto_mensal('amci')[['nivel_carreira', 'valor_total_prefeitura_atual']].rename({'valor_total_prefeitura_atual': 'valor_total_prefeitura_proposta'}, axis=1)
}

In [41]:
impactos_mensais['Equiparação com a tabela dos AMCI']

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1706403.77,situacao_atual,2103774.73,unificacao_amci,397370.96
1,2,1127593.20,situacao_atual,1333681.20,unificacao_amci,206088.00
2,3,40615.73,situacao_atual,48160.64,unificacao_amci,7544.91
3,4,21916.01,situacao_atual,26021.22,unificacao_amci,4105.21
4,5,356961.79,situacao_atual,421643.12,unificacao_amci,64681.33
5,6,851334.48,situacao_atual,1001681.02,unificacao_amci,150346.54


In [42]:

def gerar_grafico_niveis_df(df, col_valor:str, titulo:str, nome_arquivo="grafico_niveis.png"):
    
    col_nivel = 'nivel_carreira'
    
    cores = ["steelblue"] * len(df)
    indice_maximo = df[col_valor].idxmax()
    cores[indice_maximo] = "crimson"

    # Formatação para o texto sobre as barras (R$ 1.234,56)
    texto_formatado = df[col_valor].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    fig = go.Figure(data=[
        go.Bar(
            x=df[col_nivel],
            y=df[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        # Define os separadores globalmente: o primeiro é o decimal, o segundo é o de milhar
        separators=",.",
        plot_bgcolor="white",
        width=1200,
        height=600,
        xaxis=dict(
            title="Níveis de Carreira",
            tickangle=-45,
            categoryorder="array",
            categoryarray=df[col_nivel].tolist()
        ),
        yaxis=dict(
            title="Valores em R$",
            showgrid=True,
            gridcolor="lightgrey",
            # Formata os números do eixo Y com separador de milhar e 2 casas decimais
            tickformat=",2f"
        ),
        margin=dict(l=50, r=50, t=100, b=120)
    )

    fig.write_image(nome_arquivo)

    return fig

In [43]:
for nome, df in impactos_mensais.items():
    titulo = f"Custo total por nível: {nome}"
    nome_arquivo = f"grafico_niveis_{nome.replace(' ', '_').lower()}.png"
    gerar_grafico_niveis_df(df, "valor_total_prefeitura_proposta", titulo, nome_arquivo)

In [44]:
for proposta, df in impactos_mensais.items():
    valor_total = df['valor_total_prefeitura_proposta'].sum()*12
    valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"Valor total anual: {valor_formatado} - {proposta}")

Valor total anual: R$ 59.219.543,16 - Equiparação com a tabela dos AMCI
Valor total anual: R$ 54.891.933,48 - Reajuste pelo IPC-Fipe - jun. 2016
Valor total anual: R$ 49.342.033,32 - Reajuste pelo IPC-Fipe - jun. 2021
Valor total anual: R$ 49.257.899,76 - Situação atual


In [45]:
for proposta, df in impactos_mensais.items():
    try:
        valor_total = df['impacto_mensal'].sum()*12
        valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        print(f"Impacot total anual: {valor_formatado} - {proposta}")
    except KeyError:
        print(proposta, "não possui impacto")

Impacot total anual: R$ 9.961.643,40 - Equiparação com a tabela dos AMCI
Impacot total anual: R$ 5.634.033,72 - Reajuste pelo IPC-Fipe - jun. 2016
Impacot total anual: R$ 84.133,56 - Reajuste pelo IPC-Fipe - jun. 2021
Situação atual não possui impacto


In [46]:
niveis = pd.read_csv('tabelas_vencimentos_utilizadas.csv', sep=';', index_col=0)

In [47]:
niveis

,nome_tabela,nivel,vencimento
0,atual,1,13208.14
1,atual,2,14528.96
2,atual,3,14892.18
3,atual,4,15264.48
4,atual,5,15646.09
...,...,...,...
10,original_atualizada_ipc_nunes,11,21929.28
11,original_atualizada_ipc_nunes,12,24122.20
12,original_atualizada_ipc_nunes,13,25207.70
13,original_atualizada_ipc_nunes,14,26342.05


In [48]:
import plotly.graph_objects as go
import pandas as pd

def exportar_graficos_comparativos(df, col_valor="vencimento"):
    col_nivel = "nivel"
    col_tabela = "nome_tabela"
    
    # Identifica as tabelas que serão comparadas com a 'atual'
    tabelas_extras = [t for t in df[col_tabela].unique() if t != "atual"]
    
    for tabela in tabelas_extras:
        # Filtra apenas o par necessário
        df_par = df[df[col_tabela].isin(["atual", tabela])]
        
        fig = go.Figure()

        # Adiciona as barras para 'atual' e para a 'tabela' da vez
        for nome in ["atual", tabela]:
            df_sub = df_par[df_par[col_tabela] == nome]
            
            # Formatação Real (sem centavos para evitar sobreposição de texto)
            texto = df_sub[col_valor].apply(
                lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

            fig.add_trace(go.Bar(
                x=df_sub[col_nivel],
                y=df_sub[col_valor],
                name=nome.replace("_", " ").title(),
                text=texto,
                textposition="outside",
                marker_color="steelblue" if nome == "atual" else "crimson"
            ))

        fig.update_layout(
            title=dict(
                text=f"Comparativo: Atual vs {tabela.replace('_', ' ').title()}",
                x=0.5,
                font=dict(size=22)
            ),
            barmode="group",
            separators=",.",
            plot_bgcolor="white",
            width=1200,
            height=600,
            xaxis=dict(title="Nível de Carreira", type="category"),
            yaxis=dict(
                title="Vencimento (R$)",
                showgrid=True,
                gridcolor="lightgrey",
                tickformat=",0f"
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Salva cada gráfico com o nome da tabela correspondente
        nome_arquivo = f"comparativo_atual_vs_{tabela}.png"
        fig.write_image(nome_arquivo, engine="kaleido")
        print(f"Arquivo salvo: {nome_arquivo}")

# Exemplo de uso:
# exportar_graficos_comparativos(df)

In [49]:
exportar_graficos_comparativos(niveis)

/tmp/ipykernel_207658/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original.png


/tmp/ipykernel_207658/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_amci.png


/tmp/ipykernel_207658/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_ipc.png


/tmp/ipykernel_207658/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_ipc_nunes.png


In [50]:
impacto = pd.read_csv('impactos_anuais.csv', sep=';', index_col=0)

In [51]:
impacto

,nivel,vencimento
0,reajuste_ipca_gestao_nunes,84133.56
1,reajuste_ipca,5634033.72
2,reajuste_amci,9961643.40


In [52]:
impacto= impacto.rename({'nivel' : 'Proposta Reajusta', 'vencimento' : "Impacto orçamentário anualizado"}, axis=1)

In [53]:
impacto

,Proposta Reajusta,Impacto orçamentário anualizado
0,reajuste_ipca_gestao_nunes,84133.56
1,reajuste_ipca,5634033.72
2,reajuste_amci,9961643.40


In [54]:
impacto['Proposta Reajusta'] = impacto['Proposta Reajusta'].str.replace('_', ' ').str.replace('ipca', 'icp fipe').str.title()

In [55]:
impacto

,Proposta Reajusta,Impacto orçamentário anualizado
0,Reajuste Icp Fipe Gestao Nunes,84133.56
1,Reajuste Icp Fipe,5634033.72
2,Reajuste Amci,9961643.40


In [56]:
import plotly.graph_objects as go
import pandas as pd

def gerar_grafico_simples_ordenado(df, col_valor="vencimento", titulo="Comparativo de Reajustes", nome_arquivo="grafico_simples.png"):
    
    col_tabela = "Proposta Reajusta"
    
    # Ordena o DataFrame pelo valor do vencimento (menor para o maior)
    df_ordenado = df.sort_values(by=col_valor, ascending=True)

    # Formatação para o texto sobre as barras (R$ 1.234)
    texto_formatado = df_ordenado[col_valor].apply(
        lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    # Define cores: Azul para a 'atual' e Steelblue para as demais
    cores = ["steelblue" if n != "atual" else "darkblue" for n in df_ordenado[col_tabela]]

    fig = go.Figure(data=[
        go.Bar(
            x=df_ordenado[col_tabela],
            y=df_ordenado[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            font=dict(size=22)
        ),
        separators=",.",
        plot_bgcolor="white",
        width=1000,
        height=600,
        xaxis=dict(
            title="Cenários / Tabelas",
            tickangle=0 # Mantém reto para facilitar leitura se forem poucos nomes
        ),
        yaxis=dict(
            title="Vencimento (R$)",
            showgrid=True,
            gridcolor="lightgrey",
            tickformat=",0f"
        ),
        margin=dict(l=50, r=50, t=100, b=100)
    )

    fig.write_image(nome_arquivo, engine="kaleido")

    return fig

In [57]:
gerar_grafico_simples_ordenado(impacto, col_valor='Impacto orçamentário anualizado')

/tmp/ipykernel_207658/2206286243.py:52: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")
